In [15]:
import requests
import pandas as pd
import time

# 设置 CoinMarketCap API Key
API_KEY = "6caf637d-69cb-4a07-b397-a98e66b192c2"
HEADERS = {"X-CMC_PRO_API_KEY": API_KEY}

# 获取所有币种的 ID 和 Symbol
def get_all_coins():
    url = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/map"
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    return pd.DataFrame([{"id": coin["id"], "symbol": coin["symbol"]} for coin in data["data"]])

# 获取币种的 tags 信息
def get_coin_info(coin_ids):
    url = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/info"
    params = {"id": ",".join(map(str, coin_ids))}
    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    return pd.DataFrame([
        {"id": coin_id, 
         "symbol": data["data"].get(str(coin_id), {}).get("symbol", None),
         "tags": data["data"].get(str(coin_id), {}).get("tags", None)}
        for coin_id in coin_ids
    ])

# 主函数：获取全市场币种的 tags
def get_full_market_tags():
    df_coins = get_all_coins()
    
    # 批量处理，防止 API 速率限制
    batch_size = 500
    df_info_list = []
    
    for i in range(0, len(df_coins), batch_size):
        batch_ids = df_coins["id"].iloc[i:i+batch_size].tolist()
        df_info_list.append(get_coin_info(batch_ids))
        time.sleep(2)  # API 限制间隔
    
    df_info = pd.concat(df_info_list, ignore_index=True)
    
    # 合并数据
    df_final = df_coins.merge(df_info, on=["id", "symbol"], how="left")
    
    return df_final[["id", "symbol", "tags"]]

# 运行主函数
df_tags = get_full_market_tags()

# 保存结果
df_tags.to_csv("crypto_tags.csv", index=False)
print(df_tags.head())



   id symbol                                               tags
0   1    BTC  [mineable, pow, sha-256, store-of-value, state...
1   2    LTC  [mineable, pow, scrypt, medium-of-exchange, la...
2   3    NMC                 [mineable, pow, sha-256, platform]
3   4    TRC              [mineable, pow, sha-256, masternodes]
4   5    PPC  [mineable, hybrid-pow-pos, sha-256, medium-of-...


In [22]:
import pandas as pd
import ast

def classify_token(tags):
    if not isinstance(tags, list):
        return "Unknown"
    # 转换为小写
    tags_lower = [t.lower() for t in tags]
    # 如果包含 pow 或 proof of work，则归类为 brown
    if any("pow" in t or "proof of work" in t for t in tags_lower):
        return "brown"
    else:
        return "green"

# 读取 CSV 文件（假设文件名为 crypto_tags.csv）
df = pd.read_csv("crypto_tags.csv")

# 如果 tags 列是字符串形式的列表，则转换为列表
df['tags'] = df['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 应用分类函数，新增 classification 列
df["classification"] = df["tags"].apply(classify_token)
df = df.dropna()
df.drop('tags', axis=1, inplace=True)
print(df.head())

# 保存结果
df.to_csv("crypto_classification.csv", index=False)


   id symbol classification
0   1    BTC          brown
1   2    LTC          brown
2   3    NMC          brown
3   4    TRC          brown
4   5    PPC          brown


In [1]:
import pandas as pd 
import numpy as np
import statsmodels.api as sm
import glob
import os

# ========== 1. 读取并合并日度 CSV 数据 ==========
folder_path = r"D:\桌面\绿色货币\数据总库\R数据抓取加密货币数据"  # 数据文件夹路径
file_paths = sorted(glob.glob(os.path.join(folder_path, "crypto_full_20*_Daily_data.csv")))
df_daily = pd.concat((pd.read_csv(file) for file in file_paths), ignore_index=True)

# ========== 2. 定义聚合日度数据为周度数据的函数 ==========
def aggregate_daily_to_weekly(df):
    # 1) 时间转换 & 筛选日期
    df = df.rename(columns={"time_open": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    # # 如有需要可筛选时间范围，例如：
    # df = df.query('"2014-01-01" <= Date <= "2020-07-31"')
    
    # 2) 使用 ISO 8601 周规则：获取 ISO 年和 ISO 周
    iso = df["Date"].dt.isocalendar()
    df["iso_year"] = iso.year
    df["iso_week"] = iso.week

    # 3) 数据清洗：剔除缺失及异常值；同时对市值筛选（大于 1,000,000）
    df = df.dropna()
    df = df[(df["volume"] > 0) & (df["close"] > 0)]
    df = df.dropna(subset=["market_cap"])
    df = df[df["market_cap"] > 1_000_000]
    
    # 4) 计算每日价格成交额及其对数
    df["price_vol"] = df["close"] * df["volume"]
    df["log_price_vol"] = np.log(df["price_vol"])
    
    # 5) 按币种、ISO年和ISO周聚合（聚合规则与论文一致）
    agg_dict = {
        "Date": "max",          # 每只币当周最后交易日
        "close": "last",        # 本周最后一天收盘价
        "high": "max",          # 本周最高价
        "market_cap": "last",   # 本周最后一天市值
        "volume": "mean",       # 本周平均交易量
        "price_vol": "mean",    # 本周平均成交额
        "log_price_vol": "std"  # 本周对数成交额样本标准差（ddof=1）
    }
    weekly = df.groupby(["id", "iso_year", "iso_week"], as_index=False).agg(agg_dict)
    
    # 6) 重命名聚合列 & 生成对数特征
    weekly.rename(columns={
        "Date": "Date_max",
        "close": "close_last",
        "high": "high_max",
        "market_cap": "market_cap_last",
        "volume": "volume_mean",
        "price_vol": "price_vol_mean",
        "log_price_vol": "log_price_vol_std"
    }, inplace=True)
    weekly["MCAP"] = np.log(weekly["market_cap_last"])
    weekly["PRC"] = np.log(weekly["close_last"])
    weekly["MAXDPRC"] = np.log(weekly["high_max"])
    weekly["PRCVOL"] = np.log(weekly["price_vol_mean"])
    weekly.rename(columns={"log_price_vol_std": "STDPRCVOL"}, inplace=True)
    
    # 7) 计算周度对数收益率（按每只币的收盘价差分）
    weekly.sort_values(["id", "Date_max"], inplace=True)
    weekly["rtn"] = weekly.groupby("id")["close_last"].transform(lambda x: np.log(x).diff())
    
    # 8) 删除因差分产生的缺失值
    weekly.dropna(subset=["rtn"], inplace=True)
    
    # 9) 统一每周日期：使用 ISO 年、ISO 周及星期日作为代表日期
    weekly["Date"] = weekly.apply(lambda row: pd.Timestamp.fromisocalendar(int(row["iso_year"]), int(row["iso_week"]), 7), axis=1)
    
    return weekly

# ========== 3. 调用函数，得到周度数据 ==========
df_weekly = aggregate_daily_to_weekly(df_daily)


In [7]:
df_class = pd.read_csv("crypto_classification.csv")
df_class['classification'] = df_class['classification'].map({'brown': 1, 'green': 0})
merged_df = pd.merge(df_weekly, df_class[['id', 'classification']], on='id', how='inner')
merged_df

,id,iso_year,iso_week,Date_max,close_last,high_max,market_cap_last,volume_mean,price_vol_mean,STDPRCVOL,MCAP,PRC,MAXDPRC,PRCVOL,rtn,Date,classification
0,1,2014,2,2014-01-12,863.219971,1017.119995,1.057865e+10,5.961516e+07,5.184169e+10,0.385070,23.082104,6.760670,6.924730,24.671461,-0.078304,2014-01-12,1
1,1,2014,3,2014-01-19,870.960022,872.809998,1.069950e+10,2.787390e+07,2.344019e+10,0.353597,23.093463,6.769596,6.771718,23.877718,0.008927,2014-01-19,1
2,1,2014,4,2014-01-26,885.280029,897.020020,1.090280e+10,2.459300e+07,2.085859e+10,0.307821,23.112285,6.785904,6.799078,23.761032,0.016308,2014-01-26,1
3,1,2014,5,2014-02-02,825.369995,893.000000,1.019004e+10,2.715560e+07,2.195721e+10,0.522366,23.044676,6.715832,6.794587,23.812362,-0.070072,2014-02-02,1
4,1,2014,6,2014-02-09,682.900024,840.169983,8.451229e+09,4.210723e+07,3.092802e+10,0.664912,22.857578,6.526348,6.733604,24.154928,-0.189483,2014-02-09,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365003,34854,2025,1,2024-12-31,0.045424,0.047310,4.542400e+07,1.514805e+07,5.956226e+05,0.907417,17.631551,-3.091713,-3.051044,13.297363,0.912511,2025-01-05,0
365004,34926,2025,1,2024-12-31,0.373889,0.416100,3.738879e+08,4.993269e+07,1.796356e+07,0.049759,19.739467,-0.983797,-0.876829,16.703856,0.335792,2025-01-05,0
365005,35075,2024,47,2024-11-24,0.022121,0.050518,2.189953e+07,1.065456e+03,2.362282e+01,1.913931,16.901976,-3.811238,-2.985416,3.162213,0.977682,2024-11-24,0
365006,35075,2024,48,2024-12-01,0.022121,0.022121,2.189953e+07,1.476900e+03,3.267017e+01,0.000000,16.901976,-3.811238,-3.811238,3.486463,0.000000,2024-12-01,0
